# GRU Direct v4 Modular Pipeline

This notebook is a cleaner modular version of the v4 GRU direct-output model.

Main idea:
- Edit parameters in `configs/gru_config.yaml`
- Keep reusable code in `src_gru/`
- Run this notebook to train, predict, and generate a Kaggle submission

Recommended stable setting from experiments: `INPUT_LEN=90`, `N_WINDOWS=6`, with lag/rolling/calendar/price features.

In [ ]:
# If needed, install yaml parser once:
# !pip install pyyaml

import os
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
# If this notebook is not opened from the project folder, set manually:
# PROJECT_ROOT = Path(r"C:/path/to/gru_modular_pipeline")

sys.path.append(str(PROJECT_ROOT))

from src_gru.preprocessing import (
    set_seed, load_m5_raw, build_sales_matrices, encode_static_categories
)
from src_gru.features import (
    prepare_calendar_features, build_price_matrix, build_training_windows, build_prediction_block
)
from src_gru.models import build_model
from src_gru.train import (
    get_device, make_loaders, train_model, predict_future_block,
    build_submission, save_submission, ensemble_submissions
)

print('PyTorch:', torch.__version__)

In [ ]:
# =====================
# Load config
# =====================
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'gru_config.yaml'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg['training']['seed']))
DEVICE = get_device()
print('Device:', DEVICE)
print('Seed:', cfg['training']['seed'])
print('Forecast config:', cfg['forecast'])
print('Training config:', cfg['training'])
print('Model config:', cfg['model'])

In [ ]:
# =====================
# Load raw data and build base matrices
# =====================
sales_df, sample_sub, calendar, prices = load_m5_raw(cfg)
sales_values, sales_log, day_cols = build_sales_matrices(sales_df)

cat_cols = cfg['model']['static_cat_cols']
cat_matrix, cat_maps, cat_sizes = encode_static_categories(sales_df, cat_cols)

n_series, n_days = sales_values.shape
print('n_series:', n_series, '| n_days:', n_days)
print('cat_sizes:', dict(zip(cat_cols, cat_sizes)))

In [ ]:
# =====================
# Feature tables
# =====================
calendar_num, event_ids, event_map = prepare_calendar_features(calendar, cfg)
print('calendar_num:', calendar_num.shape)
print('event types:', event_map)

price_matrix = None
if cfg['features'].get('use_price_features', True):
    price_matrix = build_price_matrix(sales_df, calendar, prices, n_days=calendar_num.index.max())
    print('price_matrix:', None if price_matrix is None else price_matrix.shape)
else:
    print('Price features disabled by config.')

In [ ]:
# =====================
# Build sliding windows
# =====================
X, y, X_cat, X_num, X_future_cal, X_future_event, X_price, end_pos = build_training_windows(
    sales_values=sales_values,
    sales_log=sales_log,
    cat_matrix=cat_matrix,
    calendar_num=calendar_num,
    event_ids=event_ids,
    price_matrix=price_matrix,
    cfg=cfg,
)

feature_shapes = {
    'num_numeric_features': X_num.shape[1],
    'num_future_cal_features': X_future_cal.shape[2],
    'num_price_features': X_price.shape[1],
}
print(feature_shapes)

In [ ]:
# =====================
# DataLoaders
# =====================
train_loader, val_loader, split_info = make_loaders(
    X, y, X_cat, X_num, X_future_cal, X_future_event, X_price, end_pos, cfg
)

In [ ]:
# =====================
# Build and train model
# =====================
model = build_model(
    cat_sizes=cat_sizes,
    event_type_cardinality=len(event_map),
    feature_shapes=feature_shapes,
    cfg=cfg,
    device=DEVICE,
)
print(model)

model, history = train_model(model, train_loader, val_loader, cfg, DEVICE)
history

In [ ]:
# Optional: plot training curve
try:
    ax = history.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', figsize=(7, 4))
    ax.set_title('Training History')
    ax.set_ylabel(cfg['training']['loss'])
except Exception as e:
    print('Plot skipped:', e)

In [ ]:
# =====================
# Predict future 28-day block
# =====================
input_last = sales_log[:, -int(cfg['forecast']['input_len']):]
future_start_idx = n_days  # after d_1913 for validation forecast in M5 sales_train_validation

pred_arrays = build_prediction_block(
    sales_values=sales_values,
    sales_log_input=input_last,
    cat_matrix=cat_matrix,
    calendar_num=calendar_num,
    event_ids=event_ids,
    price_matrix=price_matrix,
    cfg=cfg,
    future_start_idx=future_start_idx,
)

pred_raw, pred_log = predict_future_block(model, pred_arrays, cfg, DEVICE)
print('pred_raw:', pred_raw.shape)
print(pd.DataFrame(pred_raw, columns=[f'F{i}' for i in range(1, 29)]).describe())
print('overall mean:', pred_raw.mean())
print('zero ratio:', (pred_raw == 0).mean())

In [ ]:
# =====================
# Build and save submission
# =====================
submission = build_submission(pred_raw, sales_df, sample_sub)
seed = cfg['training']['seed']
suffix = f"seed{seed}_{cfg['forecast']['input_len']}_{cfg['forecast']['n_windows']}"
out_path = save_submission(submission, cfg, suffix=suffix)
submission.head()

## Optional: seed ensemble

After you run the same config with different seeds, place the CSV paths below and average them.

In [ ]:
# Example ensemble usage. Uncomment and edit paths after you have multiple submissions.
# paths = [
#     'output/submission_gru_direct_config_seed42_90_6.csv',
#     'output/submission_gru_direct_config_seed2024_90_6.csv',
#     'output/submission_gru_direct_config_seed7_90_6.csv',
# ]
# ensemble = ensemble_submissions(
#     paths,
#     weights=None,  # equal average; or e.g. [0.4, 0.3, 0.3]
#     output_path='output/submission_gru_direct_seed_ensemble.csv'
# )
# ensemble.head()

## What to change in config

Most useful experiments:

```yaml
training:
  seed: 2024      # run another seed for ensemble

forecast:
  input_len: 90
  n_windows: 6

model:
  rnn_type: "gru"   # try "lstm" as an ensemble candidate

features:
  use_price_features: true
```

Based on your previous experiments, avoid changing too many things at once.